In [539]:
import re

def replace_special_characters(input_string):
    # only word and number
    result = re.sub(r'[^a-zA-Z0-9]', ' ', input_string)
    return result

number_words = {'0': 'zero', '1': 'one', '2': 'two', '3': 'three', '4': 'four',
    '5': 'five', '6': 'six', '7': 'seven', '8': 'eight', '9': 'nine'
}
def replace_first_number_with_word(s):
    match = re.match(r'(\d)', s)
    if match:
        first_digit = match.group(1)
        s = s.replace(first_digit, number_words[first_digit], 1)
    return s

def find_numbers_in_string(s):
    numbers = re.findall(r'\d+', s)
    return [int(num) for num in numbers]

# format struct name - delete space - only for words
def process_string(input_string):
    s = replace_special_characters(input_string)
    s = s.lower()
    # int -> string
    s = replace_first_number_with_word(s)
    s = s.title() # upcase
    result_string = s.replace(" ", "")
    return result_string

def col1_Presence(s):
    if s == "M":
        return "Mandatory"
    if s == "O":
        return "Optional"
    if s == "C":
        return "Conditional"
    return ""

def col2_Range(s):
    pattern = r'(\d+)\.\.<(.*?)>' # 1..<maxnoofPDUSessions>
    match = re.search(pattern, s)
    if match:
        return match.group(1), match.group(2)
    return "", ""
    
def col5_Criticality(s):
    if s == "-" or s == "–":
        return "-"
    elif s != "YES" and s != "GLOBAL" and s != "EACH":
        return "???"
    return s


In [540]:
# get id of msg - "IE type and reference" -> return [] with index = 0
# col 3
def get_format(s):
    # "9.3.1.197"
    pattern = r'\b\d+\.\d+\.\d+\.\d+\b'
    matches = re.findall(pattern, s)
    if matches:
        return matches[0] # if nill return []
    return ""

# SIZE(0..255, …) -> 0, 255
def extract_range(s):
    pattern = r'SIZE\((\d+)\.\.(\d+),?.*?\)'
    match = re.search(pattern, s)
    if match:
        return int(match.group(1)), int(match.group(2))
    else:
        return None
    
# ENUMERATED (unlicensed, nb-IoT, ..., nR-LEO,
def extract_enum_elements(input_string):
    match = re.search(r'\((.*)\)', input_string)
    if match:
        elements = match.group(1).split(',')
        cleaned_elements = [re.sub(r'\((.*?)\)', r'\1', element).strip() for element in elements]
        return cleaned_elements
    return []

def col3_Type(s):
    type = ""
    low = 0
    up = 0
    ext = ""
    choose = []

    s = s.replace("\n", "")
    s = s.replace("…", "xxx")
    s = re.sub(r'[^a-zA-Z0-9(),.{}\[\] ]+', ' ', s)

    # extensive value
    if "xxx" in s: ext = "valueExt"

    if get_format(s):  # "9.3.1.197"
        type = get_format(s)

    elif "INTEGER" in s:
        type = "integer"
        range = extract_range(s) # SIZE(0..255, …) -> 0, 255
        if "4,000,000,000,000" in s: up = "4000000000000"
        elif "INTEGER (1..30|40|" in s: pass
        elif "217-1" in s: up = "217-1"
        elif "264-1" in s: up = "264-1"
        elif "232-1" in s: up ="232-1"
        elif range: 
            low = range[0]
            up = range[1]

    elif "BITSTRING" in s: # constance size
        type = "bitstring"
        size = find_numbers_in_string(s)[0]
        low = size
        up = size
    elif "BIT STRING {" in s: # constance size
        type = "bitstring"
        size = find_numbers_in_string(s)[0]
        low = size
        up = size
    elif "BIT STRING" in s: # constance size
        type = "bitstring"
        size = find_numbers_in_string(s)
        if len(size) == 1:
            low = size[0]
            up = size[0]
        elif len(size) == 2:
            low = size[0]
            up = size[1]

    elif "OCTET STRING" in s:
        type = "octetstring"
        size = find_numbers_in_string(s)
        if len(size) == 1:
            low = size[0]
            up = size[0]
        elif len(size) == 2:
            low = size[0]
            up = size[1]

    elif "ENUMERATED" in s:
        type = "enumerated"
        choose = extract_enum_elements(s)
        up = len(choose)
        if len(choose) != 0:
            for i, id in enumerate(choose):
                if id == "xxx": up = i
                # else:
                #     str = process_string(id)
                #     choose.append(str)
    
    elif "CHOICE" in s:
        type = "choice"
        choose = extract_enum_elements(s)
        up = len(choose)
        if len(choose) != 0:
            for i, id in enumerate(choose):
                if id == "xxx": up = i
                # else: 
                #     str = process_string(id)
                #     choose.append(str)

    return type, low, up, ext, choose

In [541]:
array = []
array.append({"id": "9.2.1.1", "msg": "PDU SESSION RESOURCE SETUP REQUEST"})
array.append({"id": "9.2.1.2", "msg": "PDU SESSION RESOURCE SETUP RESPONSE"})
array.append({"id": "9.2.1.3", "msg": "PDU SESSION RESOURCE RELEASE COMMAND"})
array.append({"id": "9.2.1.4", "msg": "PDU SESSION RESOURCE RELEASE RESPONSE"})
array.append({"id": "9.2.1.5", "msg": "PDU SESSION RESOURCE MODIFY REQUEST"})
array.append({"id": "9.2.1.6", "msg": "PDU SESSION RESOURCE MODIFY RESPONSE"})
array.append({"id": "9.2.1.7", "msg": "PDU SESSION RESOURCE NOTIFY"})
array.append({"id": "9.2.1.8", "msg": "PDU SESSION RESOURCE MODIFY INDICATION"})
array.append({"id": "9.2.1.9", "msg": "PDU SESSION RESOURCE MODIFY CONFIRM"})
array.append({"id": "9.2.2.1", "msg": "INITIAL CONTEXT SETUP REQUEST"})
array.append({"id": "9.2.2.2", "msg": "INITIAL CONTEXT SETUP RESPONSE"})
array.append({"id": "9.2.2.3", "msg": "INITIAL CONTEXT SETUP FAILURE"})
array.append({"id": "9.2.2.4", "msg": "UE CONTEXT RELEASE REQUEST"})
array.append({"id": "9.2.2.5", "msg": "UE CONTEXT RELEASE COMMAND"})
array.append({"id": "9.2.2.6", "msg": "UE CONTEXT RELEASE COMPLETE"})
array.append({"id": "9.2.2.7", "msg": "UE CONTEXT MODIFICATION REQUEST"})
array.append({"id": "9.2.2.8", "msg": "UE CONTEXT MODIFICATION RESPONSE"})
array.append({"id": "9.2.2.9", "msg": "UE CONTEXT MODIFICATION FAILURE"})
array.append({"id": "9.2.2.10", "msg": "RRC INACTIVE TRANSITION REPORT"})
array.append({"id": "9.2.2.11", "msg": "CONNECTION ESTABLISHMENT INDICATION"})
array.append({"id": "9.2.2.12", "msg": "AMF CP RELOCATION INDICATION"})
array.append({"id": "9.2.2.13", "msg": "RAN CP RELOCATION INDICATION"})
array.append({"id": "9.2.2.14", "msg": "RETRIEVE UE INFORMATION"})
array.append({"id": "9.2.2.15", "msg": "UE INFORMATION TRANSFER"})
array.append({"id": "9.2.2.16", "msg": "UE CONTEXT SUSPEND REQUEST"})
array.append({"id": "9.2.2.17", "msg": "UE CONTEXT SUSPEND RESPONSE"})
array.append({"id": "9.2.2.18", "msg": "UE CONTEXT SUSPEND FAILURE"})
array.append({"id": "9.2.2.19", "msg": "UE CONTEXT RESUME REQUEST"})
array.append({"id": "9.2.2.20", "msg": "UE CONTEXT RESUME RESPONSE"})
array.append({"id": "9.2.2.21", "msg": "UE CONTEXT RESUME FAILURE"})
array.append({"id": "9.2.3.1", "msg": "HANDOVER REQUIRED"})
array.append({"id": "9.2.3.2", "msg": "HANDOVER COMMAND"})
array.append({"id": "9.2.3.3", "msg": "HANDOVER PREPARATION FAILURE"})
array.append({"id": "9.2.3.4", "msg": "HANDOVER REQUEST"})
array.append({"id": "9.2.3.5", "msg": "HANDOVER REQUEST ACKNOWLEDGE"})
array.append({"id": "9.2.3.6", "msg": "HANDOVER FAILURE"})
array.append({"id": "9.2.3.7", "msg": "HANDOVER NOTIFY"})
array.append({"id": "9.2.3.8", "msg": "PATH SWITCH REQUEST"})
array.append({"id": "9.2.3.9", "msg": "PATH SWITCH REQUEST ACKNOWLEDGE"})
array.append({"id": "9.2.3.10", "msg": "PATH SWITCH REQUEST FAILURE"})
array.append({"id": "9.2.3.11", "msg": "HANDOVER CANCEL"})
array.append({"id": "9.2.3.12", "msg": "HANDOVER CANCEL ACKNOWLEDGE"})
array.append({"id": "9.2.3.13", "msg": "UPLINK RAN STATUS TRANSFER"})
array.append({"id": "9.2.3.14", "msg": "DOWNLINK RAN STATUS TRANSFER"})
array.append({"id": "9.2.3.15", "msg": "HANDOVER SUCCESS"})
array.append({"id": "9.2.3.16", "msg": "UPLINK RAN EARLY STATUS TRANSFER"})
array.append({"id": "9.2.3.17", "msg": "DOWNLINK RAN EARLY STATUS TRANSFER"})
array.append({"id": "9.2.4.1", "msg": "PAGING"})
array.append({"id": "9.2.4.2", "msg": "MULTICAST GROUP PAGING"})
array.append({"id": "9.2.5.1", "msg": "INITIAL UE MESSAGE"})
array.append({"id": "9.2.5.2", "msg": "DOWNLINK NAS TRANSPORT"})
array.append({"id": "9.2.5.3", "msg": "UPLINK NAS TRANSPORT"})
array.append({"id": "9.2.5.4", "msg": "NAS NON DELIVERY INDICATION"})
array.append({"id": "9.2.5.5", "msg": "REROUTE NAS REQUEST"})
array.append({"id": "9.2.6.1", "msg": "NG SETUP REQUEST"})
array.append({"id": "9.2.6.2", "msg": "NG SETUP RESPONSE"})
array.append({"id": "9.2.6.3", "msg": "NG SETUP FAILURE"})
array.append({"id": "9.2.6.4", "msg": "RAN CONFIGURATION UPDATE"})
array.append({"id": "9.2.6.5", "msg": "RAN CONFIGURATION UPDATE ACKNOWLEDGE"})
array.append({"id": "9.2.6.6", "msg": "RAN CONFIGURATION UPDATE FAILURE"})
array.append({"id": "9.2.6.7", "msg": "AMF CONFIGURATION UPDATE"})
array.append({"id": "9.2.6.8", "msg": "AMF CONFIGURATION UPDATE ACKNOWLEDGE"})
array.append({"id": "9.2.6.9", "msg": "AMF CONFIGURATION UPDATE FAILURE"})
array.append({"id": "9.2.6.10", "msg": "AMF STATUS INDICATION"})
array.append({"id": "9.2.6.11", "msg": "NG RESET"})
array.append({"id": "9.2.6.12", "msg": "NG RESET ACKNOWLEDGE"})
array.append({"id": "9.2.6.13", "msg": "ERROR INDICATION"})
array.append({"id": "9.2.6.14", "msg": "OVERLOAD START"})
array.append({"id": "9.2.6.15", "msg": "OVERLOAD STOP"})
array.append({"id": "9.2.7.1", "msg": "UPLINK RAN CONFIGURATION TRANSFER"})
array.append({"id": "9.2.7.2", "msg": "DOWNLINK RAN CONFIGURATION TRANSFER"})
array.append({"id": "9.2.8.1", "msg": "WRITE REPLACE WARNING REQUEST"})
array.append({"id": "9.2.8.2", "msg": "WRITE REPLACE WARNING RESPONSE"})
array.append({"id": "9.2.8.3", "msg": "PWS CANCEL REQUEST"})
array.append({"id": "9.2.8.4", "msg": "PWS CANCEL RESPONSE"})
array.append({"id": "9.2.8.5", "msg": "PWS RESTART INDICATION"})
array.append({"id": "9.2.8.6", "msg": "PWS FAILURE INDICATION"})
array.append({"id": "9.2.9.1", "msg": "DOWNLINK UE ASSOCIATED NRPPA TRANSPORT"})
array.append({"id": "9.2.9.2", "msg": "UPLINK UE ASSOCIATED NRPPA TRANSPORT"})
array.append({"id": "9.2.9.3", "msg": "DOWNLINK NON UE ASSOCIATED NRPPA TRANSPORT"})
array.append({"id": "9.2.9.4", "msg": "UPLINK NON UE ASSOCIATED NRPPA TRANSPORT"})
array.append({"id": "9.2.10.1", "msg": "TRACE START"})
array.append({"id": "9.2.10.2", "msg": "TRACE FAILURE INDICATION"})
array.append({"id": "9.2.10.3", "msg": "DEACTIVATE TRACE"})
array.append({"id": "9.2.10.4", "msg": "CELL TRAFFIC TRACE"})
array.append({"id": "9.2.11.1", "msg": "LOCATION REPORTING CONTROL"})
array.append({"id": "9.2.11.2", "msg": "LOCATION REPORTING FAILURE INDICATION"})
array.append({"id": "9.2.11.3", "msg": "LOCATION REPORT"})
array.append({"id": "9.2.12.1", "msg": "UE TNLA BINDING RELEASE REQUEST"})
array.append({"id": "9.2.13.1", "msg": "UE RADIO CAPABILITY INFO INDICATION"})
array.append({"id": "9.2.13.2", "msg": "UE RADIO CAPABILITY CHECK REQUEST"})
array.append({"id": "9.2.13.3", "msg": "UE RADIO CAPABILITY CHECK RESPONSE"})
array.append({"id": "9.2.13.4", "msg": "UE RADIO CAPABILITY ID MAPPING REQUEST"})
array.append({"id": "9.2.13.5", "msg": "UE RADIO CAPABILITY ID MAPPING RESPONSE"})
array.append({"id": "9.2.14.1", "msg": "SECONDARY RAT DATA USAGE REPORT"})
array.append({"id": "9.2.15.1", "msg": "UPLINK RIM INFORMATION TRANSFER"})
array.append({"id": "9.2.15.2", "msg": "DOWNLINK RIM INFORMATION TRANSFER"})
array.append({"id": "9.2.16.1", "msg": "BROADCAST SESSION SETUP REQUEST"})
array.append({"id": "9.2.16.2", "msg": "BROADCAST SESSION SETUP RESPONSE"})
array.append({"id": "9.2.16.3", "msg": "BROADCAST SESSION SETUP FAILURE"})
array.append({"id": "9.2.16.4", "msg": "BROADCAST SESSION MODIFICATION REQUEST"})
array.append({"id": "9.2.16.5", "msg": "BROADCAST SESSION MODIFICATION RESPONSE"})
array.append({"id": "9.2.16.6", "msg": "BROADCAST SESSION MODIFICATION FAILURE"})
array.append({"id": "9.2.16.7", "msg": "BROADCAST SESSION RELEASE REQUEST"})
array.append({"id": "9.2.16.8", "msg": "BROADCAST SESSION RELEASE RESPONSE"})
array.append({"id": "9.2.16.9", "msg": "BROADCAST SESSION RELEASE REQUIRED"})
array.append({"id": "9.2.17.1", "msg": "DISTRIBUTION SETUP REQUEST"})
array.append({"id": "9.2.17.2", "msg": "DISTRIBUTION SETUP RESPONSE"})
array.append({"id": "9.2.17.3", "msg": "DISTRIBUTION SETUP FAILURE"})
array.append({"id": "9.2.17.4", "msg": "DISTRIBUTION RELEASE REQUEST"})
array.append({"id": "9.2.17.5", "msg": "DISTRIBUTION RELEASE RESPONSE"})
array.append({"id": "9.2.17.6", "msg": "MULTICAST SESSION ACTIVATION REQUEST"})
array.append({"id": "9.2.17.7", "msg": "MULTICAST SESSION ACTIVATION RESPONSE"})
array.append({"id": "9.2.17.8", "msg": "MULTICAST SESSION ACTIVATION FAILURE"})
array.append({"id": "9.2.17.9", "msg": "MULTICAST SESSION DEACTIVATION REQUEST"})
array.append({"id": "9.2.17.10", "msg": "MULTICAST SESSION DEACTIVATION RESPONSE"})
array.append({"id": "9.2.17.11", "msg": "MULTICAST SESSION UPDATE REQUEST"})
array.append({"id": "9.2.17.12", "msg": "MULTICAST SESSION UPDATE RESPONSE"})
array.append({"id": "9.2.17.13", "msg": "MULTICAST SESSION UPDATE FAILURE"})
array.append({"id": "9.3.1.1", "msg": "Message Type"})
array.append({"id": "9.3.1.2", "msg": "Cause"})
array.append({"id": "9.3.1.3", "msg": "Criticality Diagnostics"})
array.append({"id": "9.3.1.4", "msg": "Bit Rate"})
array.append({"id": "9.3.1.5", "msg": "Global RAN Node ID"})
array.append({"id": "9.3.1.6", "msg": "Global gNB ID"})
array.append({"id": "9.3.1.7", "msg": "NR CGI"})
array.append({"id": "9.3.1.8", "msg": "Global ng eNB ID"})
array.append({"id": "9.3.1.9", "msg": "E UTRA CGI"})
array.append({"id": "9.3.1.10", "msg": "GBR QoS Flow Information"})
array.append({"id": "9.3.1.12", "msg": "QoS Flow Level QoS Parameters"})
array.append({"id": "9.3.1.13", "msg": "QoS Flow List with Cause"})
array.append({"id": "9.3.1.14", "msg": "Trace Activation"})
array.append({"id": "9.3.1.15", "msg": "Core Network Assistance Information for RRC INACTIVE"})
array.append({"id": "9.3.1.16", "msg": "User Location Information"})
array.append({"id": "9.3.1.17", "msg": "Slice Support List"})
array.append({"id": "9.3.1.18", "msg": "Dynamic 5QI Descriptor"})
array.append({"id": "9.3.1.19", "msg": "Allocation and Retention Priority"})
array.append({"id": "9.3.1.20", "msg": "Source to Target Transparent Container"})
array.append({"id": "9.3.1.21", "msg": "Target to Source Transparent Container"})
array.append({"id": "9.3.1.22", "msg": "Handover Type"})
array.append({"id": "9.3.1.23", "msg": "MICO Mode Indication"})
array.append({"id": "9.3.1.24", "msg": "S NSSAI"})
array.append({"id": "9.3.1.25", "msg": "Target ID"})
array.append({"id": "9.3.1.26", "msg": "Emergency Fallback Indicator"})
array.append({"id": "9.3.1.27", "msg": "Security Indication"})
array.append({"id": "9.3.1.28", "msg": "Non Dynamic 5QI Descriptor"})
array.append({"id": "9.3.1.29", "msg": "Source NG RAN Node to Target NG RAN Node Transparent Container"})
array.append({"id": "9.3.1.30", "msg": "Target NG RAN Node to Source NG RAN Node Transparent Container"})
array.append({"id": "9.3.1.31", "msg": "Allowed NSSAI"})
array.append({"id": "9.3.1.32", "msg": "Relative AMF Capacity"})
array.append({"id": "9.3.1.33", "msg": "DL Forwarding"})
array.append({"id": "9.3.1.34", "msg": "DRBs to QoS Flows Mapping List"})
array.append({"id": "9.3.1.35", "msg": "Message Identifier"})
array.append({"id": "9.3.1.36", "msg": "Serial Number"})
array.append({"id": "9.3.1.37", "msg": "Warning Area List"})
array.append({"id": "9.3.1.38", "msg": "Number of Broadcasts Requested"})
array.append({"id": "9.3.1.39", "msg": "Warning Type"})
array.append({"id": "9.3.1.41", "msg": "Data Coding Scheme"})
array.append({"id": "9.3.1.42", "msg": "Warning Message Contents"})
array.append({"id": "9.3.1.43", "msg": "Broadcast Completed Area List"})
array.append({"id": "9.3.1.44", "msg": "Broadcast Cancelled Area List"})
array.append({"id": "9.3.1.45", "msg": "Number of Broadcasts"})
array.append({"id": "9.3.1.46", "msg": "Concurrent Warning Message Indicator"})
array.append({"id": "9.3.1.47", "msg": "Cancel All Warning Messages Indicator"})
array.append({"id": "9.3.1.48", "msg": "Emergency Area ID"})
array.append({"id": "9.3.1.49", "msg": "Repetition Period"})
array.append({"id": "9.3.1.50", "msg": "PDU Session ID"})
array.append({"id": "9.3.1.51", "msg": "QoS Flow Identifier"})
array.append({"id": "9.3.1.52", "msg": "PDU Session Type"})
array.append({"id": "9.3.1.53", "msg": "DRB ID"})
array.append({"id": "9.3.1.54", "msg": "Masked IMEISV"})
array.append({"id": "9.3.1.55", "msg": "New Security Context Indicator"})
array.append({"id": "9.3.1.56", "msg": "Time to Wait"})
array.append({"id": "9.3.1.57", "msg": "Global N3IWF ID"})
array.append({"id": "9.3.1.58", "msg": "UE Aggregate Maximum Bit Rate"})
array.append({"id": "9.3.1.59", "msg": "Security Result"})
array.append({"id": "9.3.1.60", "msg": "User Plane Security Information"})
array.append({"id": "9.3.1.61", "msg": "Index to RAT Frequency Selection Priority"})
array.append({"id": "9.3.1.62", "msg": "Data Forwarding Accepted"})
array.append({"id": "9.3.1.63", "msg": "Data Forwarding Not Possible"})
array.append({"id": "9.3.1.64", "msg": "Direct Forwarding Path Availability"})
array.append({"id": "9.3.1.65", "msg": "Location Reporting Request Type"})
array.append({"id": "9.3.1.66", "msg": "Area of Interest"})
array.append({"id": "9.3.1.67", "msg": "UE Presence in Area of Interest List"})
array.append({"id": "9.3.1.68", "msg": "UE Radio Capability for Paging"})
array.append({"id": "9.3.1.69", "msg": "Assistance Data for Paging"})
array.append({"id": "9.3.1.70", "msg": "Assistance Data for Recommended Cells"})
array.append({"id": "9.3.1.71", "msg": "Recommended Cells for Paging"})
array.append({"id": "9.3.1.72", "msg": "Paging Attempt Information"})
array.append({"id": "9.3.1.73", "msg": "NG RAN CGI"})
array.append({"id": "9.3.1.74", "msg": "UE Radio Capability"})
array.append({"id": "9.3.1.74a", "msg": "UE Radio Capability E UTRA Format"})
array.append({"id": "9.3.1.75", "msg": "Time Stamp"})
array.append({"id": "9.3.1.76", "msg": "Location Reporting Reference ID"})
array.append({"id": "9.3.1.77", "msg": "Data Forwarding Response DRB List"})
array.append({"id": "9.3.1.78", "msg": "Paging Priority"})
array.append({"id": "9.3.1.79", "msg": "Packet Loss Rate"})
array.append({"id": "9.3.1.80", "msg": "Packet Delay Budget"})
array.append({"id": "9.3.1.81", "msg": "Packet Error Rate"})
array.append({"id": "9.3.1.82", "msg": "Averaging Window"})
array.append({"id": "9.3.1.83", "msg": "Maximum Data Burst Volume"})
array.append({"id": "9.3.1.84", "msg": "Priority Level"})
array.append({"id": "9.3.1.85", "msg": "Mobility Restriction List"})
array.append({"id": "9.3.1.86", "msg": "UE Security Capabilities"})
array.append({"id": "9.3.1.87", "msg": "Security Key"})
array.append({"id": "9.3.1.88", "msg": "Security Context"})
array.append({"id": "9.3.1.89", "msg": "IMS Voice Support Indicator"})
array.append({"id": "9.3.1.90", "msg": "Paging DRX"})
array.append({"id": "9.3.1.91", "msg": "RRC Inactive Transition Report Request"})
array.append({"id": "9.3.1.92", "msg": "RRC State"})
array.append({"id": "9.3.1.93", "msg": "Expected UE Behaviour"})
array.append({"id": "9.3.1.94", "msg": "Expected UE Activity Behaviour"})
array.append({"id": "9.3.1.95", "msg": "UE History Information"})
array.append({"id": "9.3.1.96", "msg": "Last Visited Cell Information"})
array.append({"id": "9.3.1.97", "msg": "Last Visited NG RAN Cell Information"})
array.append({"id": "9.3.1.98", "msg": "Cell Type"})
array.append({"id": "9.3.1.99", "msg": "Associated QoS Flow List"})
array.append({"id": "9.3.1.100", "msg": "Information on Recommended Cells and RAN Nodes for Paging"})
array.append({"id": "9.3.1.101", "msg": "Recommended RAN Nodes for Paging"})
array.append({"id": "9.3.1.102", "msg": "PDU Session Aggregate Maximum Bit Rate"})
array.append({"id": "9.3.1.103", "msg": "Maximum Integrity Protected Data Rate"})
array.append({"id": "9.3.1.104", "msg": "Overload Response"})
array.append({"id": "9.3.1.105", "msg": "Overload Action"})
array.append({"id": "9.3.1.106", "msg": "Traffic Load Reduction Indication"})
array.append({"id": "9.3.1.107", "msg": "Slice Overload List"})
array.append({"id": "9.3.1.108", "msg": "RAN Status Transfer Transparent Container"})
array.append({"id": "9.3.1.109", "msg": "COUNT Value for PDCP SN Length 12"})
array.append({"id": "9.3.1.110", "msg": "COUNT Value for PDCP SN Length 18"})
array.append({"id": "9.3.1.111", "msg": "RRC Establishment Cause"})
array.append({"id": "9.3.1.112", "msg": "Warning Area Coordinates"})
array.append({"id": "9.3.1.113", "msg": "Network Instance"})
array.append({"id": "9.3.1.114", "msg": "Secondary RAT Usage Information"})
array.append({"id": "9.3.1.115", "msg": "Volume Timed Report List"})
array.append({"id": "9.3.1.116", "msg": "Redirection for Voice EPS Fallback"})
array.append({"id": "9.3.1.117", "msg": "UE Retention Information"})
array.append({"id": "9.3.1.118", "msg": "UL Forwarding"})
array.append({"id": "9.3.1.119", "msg": "CN Assisted RAN Parameters Tuning"})
array.append({"id": "9.3.1.120", "msg": "Common Network Instance"})
array.append({"id": "9.3.1.121", "msg": "Data Forwarding Response E RAB List"})
array.append({"id": "9.3.1.122", "msg": "gNB Set ID"})
array.append({"id": "9.3.1.123", "msg": "RNC ID"})
array.append({"id": "9.3.1.124", "msg": "Extended RNC ID"})
array.append({"id": "9.3.1.125", "msg": "RAT Information"})
array.append({"id": "9.3.1.126", "msg": "Extended RAT Restriction Information"})
array.append({"id": "9.3.1.127", "msg": "SgNB UE X2AP ID"})
array.append({"id": "9.3.1.128", "msg": "SRVCC Operation Possible"})
array.append({"id": "9.3.1.129", "msg": "IAB Authorized"})
array.append({"id": "9.3.1.130", "msg": "TSC Traffic Characteristics"})
array.append({"id": "9.3.1.131", "msg": "TSC Assistance Information"})
array.append({"id": "9.3.1.132", "msg": "Periodicity"})
array.append({"id": "9.3.1.133", "msg": "Burst Arrival Time"})
array.append({"id": "9.3.1.134", "msg": "Redundant QoS Flow Indicator"})
array.append({"id": "9.3.1.135", "msg": "Extended Packet Delay Budget"})
array.append({"id": "9.3.1.136", "msg": "Redundant PDU Session Information"})
array.append({"id": "9.3.1.137", "msg": "NB IoT Default Paging DRX"})
array.append({"id": "9.3.1.138", "msg": "NB IoT Paging eDRX Information"})
array.append({"id": "9.3.1.139", "msg": "NB IoT Paging DRX"})
array.append({"id": "9.3.1.140", "msg": "Enhanced Coverage Restriction"})
array.append({"id": "9.3.1.141", "msg": "Paging Assistance Data for CE Capable UE"})
array.append({"id": "9.3.1.142", "msg": "UE Radio Capability ID"})
array.append({"id": "9.3.1.143", "msg": "WUS Assistance Information"})
array.append({"id": "9.3.1.144", "msg": "UE Differentiation Information"})
array.append({"id": "9.3.1.145", "msg": "NB IoT UE Priority"})
array.append({"id": "9.3.1.146", "msg": "NR V2X Services Authorized"})
array.append({"id": "9.3.1.147", "msg": "LTE V2X Services Authorized"})
array.append({"id": "9.3.1.148", "msg": "NR UE Sidelink Aggregate Maximum Bit Rate"})
array.append({"id": "9.3.1.149", "msg": "LTE UE Sidelink Aggregate Maximum Bit Rate"})
array.append({"id": "9.3.1.150", "msg": "PC5 QoS Parameters"})
array.append({"id": "9.3.1.151", "msg": "Alternative QoS Parameters Set List"})
array.append({"id": "9.3.1.152", "msg": "Alternative QoS Parameters Set Index"})
array.append({"id": "9.3.1.153", "msg": "Alternative QoS Parameters Set Notify Index"})
array.append({"id": "9.3.1.154", "msg": "E UTRA Paging eDRX Information"})
array.append({"id": "9.3.1.155", "msg": "CE mode B Restricted"})
array.append({"id": "9.3.1.156", "msg": "CE mode B Support Indicator"})
array.append({"id": "9.3.1.157", "msg": "LTE M Indication"})
array.append({"id": "9.3.1.158", "msg": "Suspend Request Indication"})
array.append({"id": "9.3.1.159", "msg": "Suspend Response Indication"})
array.append({"id": "9.3.1.160", "msg": "UE User Plane CIoT Support Indicator"})
array.append({"id": "9.3.1.161", "msg": "Global TNGF ID"})
array.append({"id": "9.3.1.162", "msg": "Global W AGF ID"})
array.append({"id": "9.3.1.163", "msg": "Global TWIF ID"})
array.append({"id": "9.3.1.164", "msg": "W AGF User Location Information"})
array.append({"id": "9.3.1.165", "msg": "Global eNB ID"})
array.append({"id": "9.3.1.166", "msg": "UE History Information from UE"})
array.append({"id": "9.3.1.167", "msg": "MDT Configuration"})
array.append({"id": "9.3.1.168", "msg": "MDT PLMN List"})
array.append({"id": "9.3.1.169", "msg": "MDT Configuration NR"})
array.append({"id": "9.3.1.170", "msg": "MDT Configuration EUTRA"})
array.append({"id": "9.3.1.171", "msg": "M1 Configuration"})
array.append({"id": "9.3.1.172", "msg": "M4 Configuration"})
array.append({"id": "9.3.1.173", "msg": "M5 Configuration"})
array.append({"id": "9.3.1.174", "msg": "M6 Configuration"})
array.append({"id": "9.3.1.175", "msg": "M7 Configuration"})
array.append({"id": "9.3.1.176", "msg": "MDT Location Information"})
array.append({"id": "9.3.1.177", "msg": "Bluetooth Measurement Configuration"})
array.append({"id": "9.3.1.178", "msg": "WLAN Measurement Configuration"})
array.append({"id": "9.3.1.179", "msg": "Sensor Measurement Configuration"})
array.append({"id": "9.3.1.180", "msg": "Event Trigger Logged MDT Configuration"})
array.append({"id": "9.3.1.181", "msg": "NR Frequency Info"})
array.append({"id": "9.3.1.182", "msg": "Area Scope of Neighbour Cells"})
array.append({"id": "9.3.1.183", "msg": "NPN Paging Assistance Information"})
array.append({"id": "9.3.1.184", "msg": "NPN Mobility Information"})
array.append({"id": "9.3.1.185", "msg": "Cell CAG Information"})
array.append({"id": "9.3.1.186", "msg": "Target to Source Failure Transparent Container"})
array.append({"id": "9.3.1.187", "msg": "Target NG RAN Node to Source NG RAN Node Failure Transparent Container"})
array.append({"id": "9.3.1.188", "msg": "DAPS Request Information"})
array.append({"id": "9.3.1.189", "msg": "DAPS Response Information"})
array.append({"id": "9.3.1.190", "msg": "Early Status Transfer Transparent Container"})
array.append({"id": "9.3.1.191", "msg": "Extended Slice Support List"})
array.append({"id": "9.3.1.192", "msg": "UE Capability Info Request"})
array.append({"id": "9.3.1.193", "msg": "Extended RAN Node Name"})
array.append({"id": "9.3.1.194", "msg": "MICO All PLMN"})
array.append({"id": "9.3.1.195", "msg": "Source Node ID"})
array.append({"id": "9.3.1.196", "msg": "E UTRAN Composite Available Capacity Group"})
array.append({"id": "9.3.1.197", "msg": "E UTRAN Composite Available Capacity"})
array.append({"id": "9.3.1.198", "msg": "E UTRAN Cell Capacity Class Value"})
array.append({"id": "9.3.1.199", "msg": "E UTRAN Capacity Value"})
array.append({"id": "9.3.1.200", "msg": "E UTRAN Radio Resource Status"})
array.append({"id": "9.3.1.205", "msg": "NR Radio Resource Status"})
array.append({"id": "9.3.1.206", "msg": "MBS Session ID"})
array.append({"id": "9.3.1.207", "msg": "MBS Area Session ID"})
array.append({"id": "9.3.1.208", "msg": "MBS Service Area"})
array.append({"id": "9.3.1.209", "msg": "MBS Service Area information"})
array.append({"id": "9.3.1.210", "msg": "MBS Support Indicator"})
array.append({"id": "9.3.1.211", "msg": "MBS Session Setup Request List"})
array.append({"id": "9.3.1.212", "msg": "MBS Session Setup or Modify Request List"})
array.append({"id": "9.3.1.213", "msg": "MBS Session Setup Response List"})
array.append({"id": "9.3.1.214", "msg": "MBS Session Failed to Setup List"})
array.append({"id": "9.3.1.215", "msg": "MBS Session To Release List"})
array.append({"id": "9.3.1.216", "msg": "Multicast Group Paging Area"})
array.append({"id": "9.3.1.217", "msg": "MBS Session Status"})
array.append({"id": "9.3.1.218", "msg": "MRB ID"})
array.append({"id": "9.3.1.219", "msg": "MRB Progress Information"})
array.append({"id": "9.3.1.220", "msg": "Time Synchronisation Assistance Information"})
array.append({"id": "9.3.1.221", "msg": "Survival Time"})
array.append({"id": "9.3.1.222", "msg": "QMC Deactivation"})
array.append({"id": "9.3.1.223", "msg": "QMC Configuration Information"})
array.append({"id": "9.3.1.224", "msg": "UE Application Layer Measurement Configuration Information"})
array.append({"id": "9.3.1.225", "msg": "Available RAN Visible QoE Metrics"})
array.append({"id": "9.3.1.227", "msg": "NR Paging eDRX Information"})
array.append({"id": "9.3.1.228", "msg": "RedCap Indication"})
array.append({"id": "9.3.1.229", "msg": "Target NSSAI Information"})
array.append({"id": "9.3.1.230", "msg": "Target NSSAI"})
array.append({"id": "9.3.1.231", "msg": "UE Slice Maximum Bit Rate List"})
array.append({"id": "9.3.1.232", "msg": "PEIPS Assistance Information"})
array.append({"id": "9.3.1.233", "msg": "5G ProSe Authorized"})
array.append({"id": "9.3.1.234", "msg": "5G ProSe PC5 QoS Parameters"})
array.append({"id": "9.3.1.235", "msg": "Last Visited PSCell Information"})
array.append({"id": "9.3.1.236", "msg": "MBS QoS Flows To Be Setup List"})
array.append({"id": "9.3.1.237", "msg": "Reporting System"})
array.append({"id": "9.3.1.238", "msg": "TAI NSAG Support List"})
array.append({"id": "9.3.1.239", "msg": "NGAP Protocol IE Id"})
array.append({"id": "9.3.1.240", "msg": "NGAP Protocol IE Support Information"})
array.append({"id": "9.3.1.241", "msg": "NGAP Protocol IE Presence Information"})
array.append({"id": "9.3.1.242", "msg": "NGAP IE Support Information Response List"})
array.append({"id": "9.3.1.243", "msg": "MDT PLMN Modification List"})
array.append({"id": "9.3.1.244", "msg": "Excess Packet Delay Threshold Configuration"})
array.append({"id": "9.3.2.1", "msg": "QoS Flow per TNL Information List"})
array.append({"id": "9.3.2.2", "msg": "UP Transport Layer Information"})
array.append({"id": "9.3.2.3", "msg": "E RAB ID"})
array.append({"id": "9.3.2.4", "msg": "Transport Layer Address"})
array.append({"id": "9.3.2.5", "msg": "GTP TEID"})
array.append({"id": "9.3.2.6", "msg": "CP Transport Layer Information"})
array.append({"id": "9.3.2.7", "msg": "TNL Association List"})
array.append({"id": "9.3.2.8", "msg": "QoS Flow per TNL Information"})
array.append({"id": "9.3.2.9", "msg": "TNL Association Usage"})
array.append({"id": "9.3.2.10", "msg": "TNL Address Weight Factor"})
array.append({"id": "9.3.2.11", "msg": "UP Transport Layer Information Pair List"})
array.append({"id": "9.3.2.12", "msg": "UP Transport Layer Information List"})
array.append({"id": "9.3.2.13", "msg": "QoS Flow List with Data Forwarding"})
array.append({"id": "9.3.2.14", "msg": "URI"})
array.append({"id": "9.3.2.15", "msg": "MBS Session TNL Information 5GC"})
array.append({"id": "9.3.2.16", "msg": "Shared NG U Multicast TNL Information"})
array.append({"id": "9.3.2.17", "msg": "MBS Session TNL Information NG RAN"})
array.append({"id": "9.3.3.1", "msg": "AMF UE NGAP ID"})
array.append({"id": "9.3.3.2", "msg": "RAN UE NGAP ID"})
array.append({"id": "9.3.3.3", "msg": "GUAMI"})
array.append({"id": "9.3.3.4", "msg": "NAS PDU"})
array.append({"id": "9.3.3.5", "msg": "PLMN Identity"})
array.append({"id": "9.3.3.6", "msg": "SON Configuration Transfer"})
array.append({"id": "9.3.3.7", "msg": "SON Information"})
array.append({"id": "9.3.3.8", "msg": "SON Information Reply"})
array.append({"id": "9.3.3.9", "msg": "Xn TNL Configuration Info"})
array.append({"id": "9.3.3.10", "msg": "TAC"})
array.append({"id": "9.3.3.11", "msg": "TAI"})
array.append({"id": "9.3.3.12", "msg": "AMF Set ID"})
array.append({"id": "9.3.3.13", "msg": "Routing ID"})
array.append({"id": "9.3.3.14", "msg": "NRPPa PDU"})
array.append({"id": "9.3.3.15", "msg": "RAN Paging Priority"})
array.append({"id": "9.3.3.16", "msg": "EPS TAC"})
array.append({"id": "9.3.3.17", "msg": "EPS TAI"})
array.append({"id": "9.3.3.18", "msg": "UE Paging Identity"})
array.append({"id": "9.3.3.19", "msg": "AMF Pointer"})
array.append({"id": "9.3.3.20", "msg": "5G S TMSI"})
array.append({"id": "9.3.3.21", "msg": "AMF Name"})
array.append({"id": "9.3.3.22", "msg": "Paging Origin"})
array.append({"id": "9.3.3.23", "msg": "UE Identity Index Value"})
array.append({"id": "9.3.3.24", "msg": "Periodic Registration Update Timer"})
array.append({"id": "9.3.3.25", "msg": "UE associated Logical NG connection List"})
array.append({"id": "9.3.3.26", "msg": "NAS Security Parameters from NG RAN"})
array.append({"id": "9.3.3.27", "msg": "Source to Target AMF Information Reroute"})
array.append({"id": "9.3.3.28", "msg": "RIM Information Transfer"})
array.append({"id": "9.3.3.29", "msg": "RIM Information"})
array.append({"id": "9.3.3.30", "msg": "LAI"})
array.append({"id": "9.3.3.31", "msg": "Extended Connected Time"})
array.append({"id": "9.3.3.32", "msg": "End Indication"})
array.append({"id": "9.3.3.33", "msg": "Inter system SON Configuration Transfer"})
array.append({"id": "9.3.3.34", "msg": "Inter system SON Information"})
array.append({"id": "9.3.3.35", "msg": "SON Information Report"})
array.append({"id": "9.3.3.36", "msg": "Inter system SON Information Report"})
array.append({"id": "9.3.3.37", "msg": "Failure Indication"})
array.append({"id": "9.3.3.38", "msg": "Inter system Failure Indication"})
array.append({"id": "9.3.3.39", "msg": "HO Report"})
array.append({"id": "9.3.3.40", "msg": "Inter system HO Report"})
array.append({"id": "9.3.3.41", "msg": "UE RLF Report Container"})
array.append({"id": "9.3.3.42", "msg": "NID"})
array.append({"id": "9.3.3.43", "msg": "CAG ID"})
array.append({"id": "9.3.3.44", "msg": "NPN Support"})
array.append({"id": "9.3.3.45", "msg": "Allowed PNI NPN List"})
array.append({"id": "9.3.3.46", "msg": "NPN Access Information"})
array.append({"id": "9.3.3.47", "msg": "Cell CAG List"})
array.append({"id": "9.3.3.48", "msg": "UL CP Security Information"})
array.append({"id": "9.3.3.49", "msg": "DL CP Security Information"})
array.append({"id": "9.3.3.50", "msg": "Configured TAC Indication"})
array.append({"id": "9.3.3.51", "msg": "Extended AMF Name"})
array.append({"id": "9.3.3.52", "msg": "Extended UE Identity Index Value"})
array.append({"id": "9.3.3.53", "msg": "NR NTN TAI Information"})
array.append({"id": "9.3.3.54", "msg": "Inter system SON Information Request"})
array.append({"id": "9.3.3.55", "msg": "Inter system SON Information Reply"})
array.append({"id": "9.3.3.56", "msg": "Inter system Cell Activation Request"})
array.append({"id": "9.3.3.57", "msg": "Inter system Cell State Indication"})
array.append({"id": "9.3.3.58", "msg": "Inter system Cell Activation Reply"})
array.append({"id": "9.3.3.59", "msg": "Inter system Resource Status Request"})
array.append({"id": "9.3.3.60", "msg": "Inter system Resource Status Report"})
array.append({"id": "9.3.3.61", "msg": "Inter system Resource Status Reply"})
array.append({"id": "9.3.3.62", "msg": "Hashed UE Identity Index Value"})
array.append({"id": "9.3.4.1", "msg": "PDU Session Resource Setup Request Transfer"})
array.append({"id": "9.3.4.2", "msg": "PDU Session Resource Setup Response Transfer"})
array.append({"id": "9.3.4.3", "msg": "PDU Session Resource Modify Request Transfer"})
array.append({"id": "9.3.4.4", "msg": "PDU Session Resource Modify Response Transfer"})
array.append({"id": "9.3.4.5", "msg": "PDU Session Resource Notify Transfer"})
array.append({"id": "9.3.4.6", "msg": "PDU Session Resource Modify Indication Transfer"})
array.append({"id": "9.3.4.7", "msg": "PDU Session Resource Modify Confirm Transfer"})
array.append({"id": "9.3.4.8", "msg": "Path Switch Request Transfer"})
array.append({"id": "9.3.4.9", "msg": "Path Switch Request Acknowledge Transfer"})
array.append({"id": "9.3.4.10", "msg": "Handover Command Transfer"})
array.append({"id": "9.3.4.11", "msg": "Handover Request Acknowledge Transfer"})
array.append({"id": "9.3.4.12", "msg": "PDU Session Resource Release Command Transfer"})
array.append({"id": "9.3.4.13", "msg": "PDU Session Resource Notify Released Transfer"})
array.append({"id": "9.3.4.14", "msg": "Handover Required Transfer"})
array.append({"id": "9.3.4.15", "msg": "Path Switch Request Setup Failed Transfer"})
array.append({"id": "9.3.4.16", "msg": "PDU Session Resource Setup Unsuccessful Transfer"})
array.append({"id": "9.3.4.17", "msg": "PDU Session Resource Modify Unsuccessful Transfer"})
array.append({"id": "9.3.4.18", "msg": "Handover Preparation Unsuccessful Transfer"})
array.append({"id": "9.3.4.19", "msg": "Handover Resource Allocation Unsuccessful Transfer"})
array.append({"id": "9.3.4.20", "msg": "Path Switch Request Unsuccessful Transfer"})
array.append({"id": "9.3.4.21", "msg": "PDU Session Resource Release Response Transfer"})
array.append({"id": "9.3.4.22", "msg": "PDU Session Resource Modify Indication Unsuccessful Transfer"})
array.append({"id": "9.3.4.23", "msg": "Secondary RAT Data Usage Report Transfer"})
array.append({"id": "9.3.4.24", "msg": "UE Context Resume Request Transfer"})
array.append({"id": "9.3.4.25", "msg": "UE Context Resume Response Transfer"})
array.append({"id": "9.3.4.26", "msg": "UE Context Suspend Request Transfer"})
array.append({"id": "9.3.5.3", "msg": "MBS Session Setup or Modification Request Transfer"})
array.append({"id": "9.3.5.5", "msg": "MBS Session Setup or Modification Response Transfer"})
array.append({"id": "9.3.5.6", "msg": "MBS Session Setup or Modification Failure Transfer"})
array.append({"id": "9.3.5.7", "msg": "MBS Distribution Setup Request Transfer"})
array.append({"id": "9.3.5.8", "msg": "MBS Distribution Setup Response Transfer"})
array.append({"id": "9.3.5.9", "msg": "MBS Distribution Setup Unsuccessful Transfer"})
array.append({"id": "9.3.5.10", "msg": "MBS Distribution Release Request Transfer"})
array.append({"id": "9.3.5.11", "msg": "Multicast Session Activation Request Transfer"})
array.append({"id": "9.3.5.12", "msg": "Multicast Session Deactivation Request Transfer"})
array.append({"id": "9.3.5.13", "msg": "Multicast Session Update Request Transfer"})
array.append({"id": "9.3.5.14", "msg": "MBS Session Release Response Transfer"})

In [542]:
import shutil, os

def delete_all_in_folder(folder_path):
    # Kiểm tra xem thư mục có tồn tại không
    if os.path.exists(folder_path):
        # Xóa tất cả nội dung trong thư mục
        shutil.rmtree(folder_path)
        # Tạo lại thư mục rỗng nếu cần thiết
        os.makedirs(folder_path)
        print(f"All contents in the folder '{folder_path}' have been deleted.")
    else:
        print(f"The folder '{folder_path}' does not exist.")

def delete_all_except(filepath, folder_path):
    # Duyệt qua tất cả các tập tin và thư mục trong folder_path
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        # Kiểm tra nếu filename không phải là file cần giữ lại
        if filename != filepath:
            # Kiểm tra nếu là một file và xóa nó
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            # Kiểm tra nếu là một thư mục và xóa nó
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)

In [543]:
from docx import Document
msg_list = {}
doc = Document('file/test.docx')
tables = doc.tables

# load table to msg name(array)
arr_index = 0
count5 = 0
count7 = 0
for i, table in enumerate(tables):
    if i < 4 or len(table.columns) == 2:
        continue
    array[arr_index]["tb_index"] = i
    array[arr_index]["tb"] = table
    if len(table.columns) == 5:
        count5 += 1
    if len(table.columns) == 7:
        count7 += 1
    arr_index += 1

print(count5, count7, arr_index)

300 173 473


In [544]:
def get_cells_7_cols(cells):
    iei = cells[0].text
    level = iei.count('>')
    iei = process_string(iei)
    iei = replace_first_number_with_word(iei)
    presence = col1_Presence(cells[1].text)
    rangeLow, rangeUp = col2_Range(cells[2].text)
    type, low, up, ext, choose = col3_Type(cells[3].text)
    if type == []:
        type = iei
    else:
        for a in array:
            if a["id"] == type:
                type = process_string(a["msg"])
                break
        if type == []: type = "???"
    criticality = col5_Criticality(cells[5].text)
    assigned = cells[6].text
    return {
        "iei" : iei,
        "level": str(level),
        "presence" : presence,
        "range" : {
            "low": rangeLow,
            "up": rangeUp
        },
        "typeCell" : {
            "type": type,
            "low": str(low),
            "up": str(up),
            "ext": ext,
            "choose": choose
        },
        "criticality": criticality,
        "assigned": assigned
    }

def get_cells_5_cols(cells):
    iei = cells[0].text
    level = iei.count('>')
    iei = process_string(iei)
    iei = replace_first_number_with_word(iei)
    presence = col1_Presence(cells[1].text)
    rangeLow, rangeUp = col2_Range(cells[2].text)
    type, low, up, ext, choose = col3_Type(cells[3].text)
    if type == []:
        type = iei
    else:
        for a in array:
            if a["id"] == type:
                type = process_string(a["msg"])
                break
        if type == []: type = "???"
    return {
        "iei" : iei,
        "level": str(level),
        "presence" : presence,
        "range" : {
            "low": rangeLow,
            "up": rangeUp
        },
        "typeCell" : {
            "type": type,
            "low": str(low),
            "up": str(up),
            "ext": ext,
            "choose": choose
        },
    }

In [545]:
def write_file(file, string):
    file.write(string)

def write_cells_to_file_7(file, name, cells):
    choose = cells["typeCell"]["choose"]
    chooseStr = ', '.join(choose)
    write_file(file, name + ".append({ \"iei\" : \"" + cells["iei"] + \
        "\", \"level\" : \"" + cells["level"] + \
        "\", \"presence\" : \"" + cells["presence"] + \
        "\", \"range-low\" : \"" + cells["range"]["low"] + \
        "\", \"range-up\" : \"" + cells["range"]["up"] + \
        "\", \"type-type\" : \"" + cells["typeCell"]["type"] + \
        "\", \"type-low\" : \"" + cells["typeCell"]["low"] + \
        "\", \"type-up\" : \"" + cells["typeCell"]["up"] + \
        "\", \"type-ext\" : \"" + cells["typeCell"]["ext"] + \
        "\", \"type-choose\" : \"" + chooseStr + \
        "\", \"criticality\" : \"" + cells["criticality"] + \
        "\", \"assigned\" : \"" + cells["assigned"] + \
        "\", \"check\" : \"" + "" + "\"})\n")

def write_cells_to_file_5(file, name, cells):
    choose = cells["typeCell"]["choose"]
    chooseStr = ', '.join(choose)
    write_file(file, name + ".append({ \"iei\" : \"" + cells["iei"] + \
        "\", \"level\" : \"" + cells["level"] + \
        "\", \"presence\" : \"" + cells["presence"] + \
        "\", \"range-low\" : \"" + cells["range"]["low"] + \
        "\", \"range-up\" : \"" + cells["range"]["up"] + \
        "\", \"type-type\" : \"" + cells["typeCell"]["type"] + \
        "\", \"type-low\" : \"" + cells["typeCell"]["low"] + \
        "\", \"type-up\" : \"" + cells["typeCell"]["up"] + \
        "\", \"type-ext\" : \"" + cells["typeCell"]["ext"] + \
        "\", \"type-choose\" : \"" + chooseStr + \
        "\", \"check\" : \"" + "" + "\"})\n")

Cache file

In [546]:
# folder_path = './cacheStruct'
# delete_all_in_folder(folder_path)
# folder_path = './ie'
# delete_all_in_folder(folder_path)

# for a in array:
#     table = a["tb"]
#     msg = process_string(a["msg"])
#     f = open("cacheStruct/" + msg + ".py", 'w')
#     f.write("ies = []\n")
#     for row in table.rows[1:]:
#         if len(row.cells) == 5:
#             write_cells_to_file_5(f, "ies", get_cells_5_cols(row.cells))
#         if len(row.cells) == 7:
#             write_cells_to_file_7(f, "ies", get_cells_7_cols(row.cells))
#     f.close()

Gen struct

In [547]:
print(array[0])
print()

{'id': '9.2.1.1', 'msg': 'PDU SESSION RESOURCE SETUP REQUEST', 'tb_index': 4, 'tb': <docx.table.Table object at 0x7897a8306ae0>}



In [548]:
defined_struct = {}

def gen_struct(ies, name):
    maxlv = -1
    for i, ie in enumerate(ies):
        if ie["range-low"] == "" and ie["range-up"] == "" and ie["type-type"] == "":
            ies[i]["type-type"] = ies[i]["iei"]
        if int(ie["level"]) > maxlv:
            maxlv = int(ie["level"])

    nameStr = name
    struct = []

    for lv in range(-1, maxlv):
        for i, ie in enumerate(ies):
            if int(ie["level"]) == lv+1 and ies[i]["check"] == "":
                ies[i]["check"] = "structed"
                struct.append(ie)
            if int(ie["level"]) < lv: continue
            if ie["iei"].startswith("CHOICE"):
                j = i
                while j < len(ies):
                    j += 1
                    subLv = int(ies[j]["level"])
                    if subLv == (lv+1) and ies[j]["check"] != "structed":
                        ies[j]["check"] = "structed"
                        struct.append(ies[j])
                    elif subLv == lv:
                        continue
            if nameStr not in defined_struct and len(struct) > 0:
                defined_struct[nameStr] = struct
            if lv == len(ies):
                break

In [ ]:
import importlib

folder_path = './cacheStruct'

for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    if os.path.isfile(file_path):  # Kiểm tra xem có phải là tệp không
        spec = importlib.util.spec_from_file_location("extend", file_path)
        extend = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(extend)

        ies = extend.ies
        filename = filename.replace(".py", "")
        if filename == "UeApplicationLayerMeasurementConfigurationInformation":
        # if filename == "Cause":
            gen_struct(ies, filename)

        # str_struct = string_struct_go(ies, filename)
        # go_file = open("./ie/" + filename + ".go", 'w')
        # write_file(go_file, str_struct)
        # go_file.close()
defined_struct